In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from huggingface_hub import hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

DATA_DIR = "/content/brain-tumor-mri-dataset"

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "Training"),
    transform=transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "Testing"),
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4
)

print("Classes:", train_dataset.classes)
print("Train :", len(train_dataset))
print("Test  :", len(test_dataset))

# RadImageNet weight from labrasool
model_path = hf_hub_download(
    repo_id="Lab-Rasool/RadImageNet",
    filename="DenseNet121.pt"
)

state_dict = torch.load(
    model_path,
    map_location=device,
    weights_only=True
)

# DenseNet121 architecture
model = models.densenet121(weights=None)

new_state_dict = {}
for key, value in state_dict.items():
    if key.startswith('backbone.0.'):
        new_key = key.replace('backbone.0.', 'features.')
        new_state_dict[new_key] = value
    else:
        new_state_dict[key] = value

# Load weights
model.load_state_dict(new_state_dict, strict=False)

model.classifier = nn.Linear(
    model.classifier.in_features,
    4
)

for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)
print("Classifier:", model.classifier)

# test
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Input :", images.shape)
print("Output:", outputs.shape)
print("Labels:", labels.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

loss = criterion(outputs, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())
print("all good")

In [ ]:
import time

epochs = 50
print(f"training {epochs} epoch")
start_time = time.time()

model.train()
for epoch in range(epochs):
    epoch_start = time.time()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_time = time.time() - epoch_start
    avg_loss = running_loss / len(train_loader)

    # progres per 5 epoch
    if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | ET: {epoch_time:.2f}s")

total_time = time.time() - start_time
print(f"\n{epochs} epoch done")
print(f"Total ET: {total_time:.2f}s ({total_time/60:.2f} mins)")
print(f"avg ET per epoch: {total_time/epochs:.2f}s")